In [19]:
import pandas as pd
import numpy as np
from datetime import datetime
import os

# -----------------------------
# USER INPUTS
# -----------------------------
BASE_DIR = "/Users/larasch/Documents/UCB_postdoc/Research/Mexico_heat_vulnerability/data"
raw_hw_path = os.path.join(BASE_DIR, "updated_temp_data/Heatwave_Indicators_Mexico_1998_2022.csv")
heatwave_summary_path = os.path.join(BASE_DIR, "updated_temp_data/Heatwave_Summary_ByPeriod_Municipality.csv")
cc_results_path       = os.path.join(BASE_DIR, "casecrossover_data/heatwave_case_crossover_all_periods.csv")
period_deaths_path    = os.path.join(BASE_DIR, "death_data/summaries/deaths_by_period_muni.csv")

# Define all heatwave metrics you want to process
HW_VARIABLES = ["hw_99_max_1","hw_95_max_1"]
CC_TYPE = "Same Day"

# -----------------------------
# STEP 0: CREATE PERIOD-LEVEL DEATHS DATASET
# -----------------------------
death_dir = os.path.join(BASE_DIR, "death_data/summaries")
period_files = [f for f in os.listdir(death_dir) if f.startswith("combined_sum_") and f.endswith(".csv")]

all_period_deaths = []

for file in period_files:
    file_path = os.path.join(death_dir, file)
    df = pd.read_csv(file_path)

    # Extract period from filename
    period_name = file.split("combined_sum_")[1].replace(".csv", "")
    period_name = period_name.replace("_", "-")  # e.g., 1998_2002 -> 1998-2002

    # Create municipality code
    df['municipality'] = "MX" + df['ENT_OCURR'].astype(str).str.zfill(2) + df['MUN_OCURR'].astype(str).str.zfill(3)

    # Sum deaths across SEXO groups
    df['deaths_total'] = df[['Group_A_SEXO','Group_B_SEXO','Group_C_SEXO']].sum(axis=1)

    # Aggregate by municipality
    df_sum = df.groupby(['ENT_OCURR', 'MUN_OCURR', 'municipality'], as_index=False)['deaths_total'].sum()
    df_sum['year_period'] = period_name

    all_period_deaths.append(df_sum)

# Combine all periods
deaths_by_period = pd.concat(all_period_deaths, ignore_index=True)
os.makedirs(os.path.dirname(period_deaths_path), exist_ok=True)
deaths_by_period.to_csv(period_deaths_path, index=False)
print(f"Saved period-level deaths dataset to: {period_deaths_path}")

# -----------------------------
# STEP 1: LOAD HEATWAVE DATA
# -----------------------------
df = pd.read_csv(raw_hw_path)

def map_year_to_period(year):
    year = int(year)
    if 1998 <= year <= 2002:
        return "1998-2002"
    elif 2003 <= year <= 2007:
        return "2003-2007"
    elif 2008 <= year <= 2012:
        return "2008-2012"
    elif 2013 <= year <= 2017:
        return "2013-2017"
    elif 2018 <= year <= 2022:
        return "2018-2022"
    else:
        return str(year)

df['year_period'] = df['year'].apply(map_year_to_period)

# Heatwave columns
hw_cols = [c for c in df.columns if c.startswith("hw_")]

# Aggregate heatwave by municipality × period
summary = df.groupby(["year_period", "municipality"], as_index=False)[hw_cols].sum()
summary = summary.sort_values(["year_period","municipality"]).reset_index(drop=True)
summary.to_csv(heatwave_summary_path, index=False)
print(f"Saved heatwave summary to: {heatwave_summary_path}")

# -----------------------------
# STEP 2: LOAD CC RESULTS AND PERIOD DEATHS
# -----------------------------
hw = summary.copy()
hw["ENT_OCURR"] = hw["municipality"].str[2:4].astype(int)  # infer state from municipality code

cc = pd.read_csv(cc_results_path)
dp = pd.read_csv(period_deaths_path)

# -----------------------------
# STEP 3: COMPUTE ATTRIBUTABLE DEATHS
# -----------------------------
def period_to_num_days(period: str) -> int:
    start_year, end_year = map(int, period.split("-"))
    start = datetime(start_year, 1, 1)
    end   = datetime(end_year, 12, 31)
    return (end - start).days + 1

def paf_from_or(or_value: float) -> float:
    if pd.isna(or_value) or or_value <= 0:
        return np.nan
    return 1.0 - (1.0 / or_value)

all_results = []

for HW_VARIABLE in HW_VARIABLES:
    print(f"Processing {HW_VARIABLE}...")

    cc_filt = cc.query("variable == @HW_VARIABLE and type == @CC_TYPE").copy()
    cc_filt = cc_filt[cc_filt["ENT_OCURR"] != "All"].copy()
    cc_filt["ENT_OCURR"] = cc_filt["ENT_OCURR"].astype(int)

    # In heatwave summary (hw)
    hw['year_period'] = hw['year_period'].str.replace("_", "-").str.strip()

    # In case-crossover results (cc)
    cc_filt['year_period'] = cc_filt['year_period'].str.replace("_", "-").str.strip()

    # Ensure ENT_OCURR is integer in both
    hw['ENT_OCURR'] = hw['ENT_OCURR'].astype(int)
    cc_filt['ENT_OCURR'] = cc_filt['ENT_OCURR'].astype(int)

    # Now you can merge safely
    merged = hw[['municipality','ENT_OCURR','year_period'] + HW_VARIABLES].merge(
        cc_filt[['ENT_OCURR','year_period','or']],
        on=['ENT_OCURR','year_period'],
        how='left'
    )
    merged["PAF"] = merged["or"].apply(paf_from_or)

    # Merge with deaths
    merged = merged.merge(dp[["municipality","year_period","deaths_total"]], on=["municipality","year_period"], how="left")
    merged["period_days"] = merged["year_period"].apply(period_to_num_days)
    merged["deaths_exposed_est"] = merged[HW_VARIABLE] / merged["period_days"] * merged["deaths_total"]
    merged["AN_deaths_est"] = merged["deaths_exposed_est"] * merged["PAF"]

    merged["hw_definition"] = HW_VARIABLE
    all_results.append(merged[["ENT_OCURR","year_period","hw_definition","AN_deaths_est"]])

df_all = pd.concat(all_results, ignore_index=True)

# Aggregate to state × period × HW definition
state_period = df_all.groupby(["ENT_OCURR","year_period","hw_definition"], as_index=False)["AN_deaths_est"].sum()

out_path = os.path.join(BASE_DIR, "Attributable_number/attributable_numbers_by_state_period.csv")
os.makedirs(os.path.dirname(out_path), exist_ok=True)
state_period.to_csv(out_path, index=False)
print(f" Saved state-period totals for all HW definitions to: {out_path}")
print(state_period.head(12))


Saved period-level deaths dataset to: /Users/larasch/Documents/UCB_postdoc/Research/Mexico_heat_vulnerability/data/death_data/summaries/deaths_by_period_muni.csv
Saved heatwave summary to: /Users/larasch/Documents/UCB_postdoc/Research/Mexico_heat_vulnerability/data/updated_temp_data/Heatwave_Summary_ByPeriod_Municipality.csv
Processing hw_99_max_1...
Processing hw_95_max_1...
 Saved state-period totals for all HW definitions to: /Users/larasch/Documents/UCB_postdoc/Research/Mexico_heat_vulnerability/data/Attributable_number/attributable_numbers_by_state_period.csv
    ENT_OCURR year_period hw_definition  AN_deaths_est
0           1   1998-2002   hw_95_max_1     -29.249217
1           1   1998-2002   hw_99_max_1     -15.320439
2           1   2003-2007   hw_95_max_1      58.932491
3           1   2003-2007   hw_99_max_1      42.179570
4           1   2008-2012   hw_95_max_1     108.523600
5           1   2008-2012   hw_99_max_1       8.100007
6           1   2013-2017   hw_95_max_1     